<a href="https://colab.research.google.com/github/pranatixsharma/Masculine_defaults_Indian_youtube/blob/main/gender_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gender Detection Pipeline


In [ ]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print("GPU available:", gpus)
if not gpus:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU, then restart.")

GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


### CELL 2: Install dependencies

In [ ]:
!pip install inaSpeechSegmenter -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.8/249.8 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 110.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.0/90.0 kB 9.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which is incompatible.


### CELL 3: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### CELL 4: Configuration - EDIT THESE PATHS

In [ ]:
import os

# Folder in your Drive where all the mp3 audio files are (or subfolders of them)
AUDIO_DIR = '/content/drive/MyDrive/colab_notebooks/masculine_default/audio'

# Local Colab disk (fast, temporary) for the trimmed 30-second clips
TRIMMED_DIR = '/content/trimmed_audio'

# Where results and errors get saved (on Drive, so they survive disconnects)
RESULTS_CSV = '/content/drive/MyDrive/colab_notebooks/masculine_default/gender_results.csv'
ERROR_LOG = '/content/drive/MyDrive/colab_notebooks/masculine_default/gender_errors.csv'

CLIP_SECONDS = 30  # matches the paper's methodology (r=0.79-0.82 correlation with full episode)

os.makedirs(TRIMMED_DIR, exist_ok=True)

### CELL 5: List files & resume support

In [ ]:
import glob
import pandas as pd

all_files = glob.glob(os.path.join(AUDIO_DIR, '**', '*.mp3'), recursive=True)
print(f"Found {len(all_files)} audio files")

if os.path.exists(RESULTS_CSV):
    done_df = pd.read_csv(RESULTS_CSV)
    done_ids = set(done_df['file'])
    print(f"Resuming: {len(done_ids)} files already processed, skipping them")
else:
    done_ids = set()

files_to_process = [f for f in all_files if os.path.basename(f) not in done_ids]
print(f"{len(files_to_process)} files left to process")

Found 9917 audio files
Resuming: 9908 files already processed, skipping them
19 files left to process


### CELL 6: Trim to first 30s (parallel, fast)

In [ ]:
import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

def trim_audio(src_path):
    filename = os.path.basename(src_path)
    dst_path = os.path.join(TRIMMED_DIR, filename)
    if os.path.exists(dst_path):
        return filename, True, None
    try:
        # -c copy = stream copy, NO re-encoding = very fast (just cuts the container)
        cmd = ['ffmpeg', '-y', '-i', src_path, '-t', str(CLIP_SECONDS), '-c', 'copy', dst_path]
        result = subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, timeout=30)
        if result.returncode != 0:
            # fallback: some files can't be stream-copied cleanly, so re-encode instead
            cmd2 = ['ffmpeg', '-y', '-i', src_path, '-t', str(CLIP_SECONDS), dst_path]
            result2 = subprocess.run(cmd2, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, timeout=60)
            if result2.returncode != 0:
                return filename, False, result2.stderr.decode(errors='ignore')[:300]
        return filename, True, None
    except Exception as e:
        return filename, False, str(e)

trim_errors = []
with ThreadPoolExecutor(max_workers=8) as executor:
    futures = {executor.submit(trim_audio, f): f for f in files_to_process}
    for future in tqdm(as_completed(futures), total=len(futures), desc="Trimming audio"):
        filename, success, err = future.result()
        if not success:
            trim_errors.append({'file': filename, 'stage': 'trim', 'error': err})

print(f"Trimming done. {len(trim_errors)} failures.")
if trim_errors:
    pd.DataFrame(trim_errors).to_csv(ERROR_LOG, mode='a', index=False,
                                      header=not os.path.exists(ERROR_LOG))

Trimming audio: 100%|██████████| 19/19 [00:00<00:00, 21.44it/s]


Trimming done. 1 failures.


### CELL 7: Load the gender detection model

In [ ]:
from inaSpeechSegmenter import Segmenter

seg = Segmenter(vad_engine='smn', detect_gender=True)
print("Model loaded. Ready to run on GPU.")

3244808/3244808 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/reshape.py:38: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


6040200/6040200 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Model loaded. Ready to run on GPU.


### CELL 8: Run gender detection with checkpointing

In [ ]:
import time

CHECKPOINT_EVERY = 50

# If True, prints the full segment-by-segment breakdown for every file (like your example).
# If the Colab tab starts feeling slow/laggy after a few thousand files, set this to False -
# every detail still gets saved to VERBOSE_LOG_FILE either way, you'd just stop watching it live.
PRINT_TO_SCREEN = True
VERBOSE_LOG_FILE = '/content/drive/MyDrive/colab_notebooks/masculine_default/gender_detection_log.txt'

results_buffer = []
error_buffer = []

trimmed_files = [
    os.path.join(TRIMMED_DIR, os.path.basename(f)) for f in files_to_process
    if os.path.exists(os.path.join(TRIMMED_DIR, os.path.basename(f)))
]

def get_display_name(filename):
    # filenames look like "channelName__videoID.mp3" -> just show the channel name
    name = filename.rsplit('.', 1)[0]
    return name.split('__')[0] if '__' in name else name

def log_line(f_handle, text):
    if PRINT_TO_SCREEN:
        print(text)
    f_handle.write(text + '\n')

start_time = time.time()
log_f = open(VERBOSE_LOG_FILE, 'a', encoding='utf-8')

for i, path in enumerate(tqdm(trimmed_files, desc="Gender detection")):
    filename = os.path.basename(path)
    try:
        segments = seg(path)  # list of (label, start_time, end_time)
        male_time = sum(end - start for label, start, end in segments if label == 'male')
        female_time = sum(end - start for label, start, end in segments if label == 'female')
        total_speech = male_time + female_time

        if total_speech == 0:
            dominant = 'unknown'
        else:
            dominant = 'male' if male_time >= female_time else 'female'
        male_pct = (male_time / total_speech * 100) if total_speech > 0 else 0.0
        female_pct = (female_time / total_speech * 100) if total_speech > 0 else 0.0

        # ---- print/log in the segment-by-segment format you asked for ----
        log_line(log_f, f"\nProcessing: {get_display_name(filename)}")
        log_line(log_f, "  Segments:")
        for label, start, end in segments:
            log_line(log_f, f"    {label:<12} {start:.1f}s -> {end:.1f}s ({end - start:.1f}s)")
        log_line(log_f, "  Summary:")
        log_line(log_f, f"    Male   : {male_time:.1f}s ({male_pct:.1f}%)")
        log_line(log_f, f"    Female : {female_time:.1f}s ({female_pct:.1f}%)")
        log_line(log_f, f"    Result : {dominant.upper()}")

        results_buffer.append({
            'file': filename,
            'male_seconds': round(male_time, 2),
            'female_seconds': round(female_time, 2),
            'dominant_gender': dominant
        })
    except Exception as e:
        error_buffer.append({'file': filename, 'stage': 'gender_detection', 'error': str(e)[:300]})
        log_line(log_f, f"\nProcessing: {get_display_name(filename)}  -->  ERROR: {str(e)[:150]}")

    # save progress periodically so a crash/disconnect doesn't lose everything
    if (i + 1) % CHECKPOINT_EVERY == 0:
        log_f.flush()
        if results_buffer:
            pd.DataFrame(results_buffer).to_csv(RESULTS_CSV, mode='a', index=False,
                                                 header=not os.path.exists(RESULTS_CSV))
            results_buffer = []
        if error_buffer:
            pd.DataFrame(error_buffer).to_csv(ERROR_LOG, mode='a', index=False,
                                               header=not os.path.exists(ERROR_LOG))
            error_buffer = []

# flush anything left in the buffer at the end
if results_buffer:
    pd.DataFrame(results_buffer).to_csv(RESULTS_CSV, mode='a', index=False,
                                         header=not os.path.exists(RESULTS_CSV))
if error_buffer:
    pd.DataFrame(error_buffer).to_csv(ERROR_LOG, mode='a', index=False,
                                       header=not os.path.exists(ERROR_LOG))

elapsed = time.time() - start_time
log_f.close()
print(f"\nDone. Processed {len(trimmed_files)} files in {elapsed/60:.1f} minutes")
if trimmed_files:
    print(f"Average: {elapsed/len(trimmed_files):.2f} sec/file")

Gender detection:   0%|          | 0/18 [00:00<?, ?it/s]

47/47 - 7s - 152ms/step
10/10 - 3s - 258ms/step


Gender detection:   6%|▌         | 1/18 [00:14<04:05, 14.42s/it]


Processing: DevikaGupta
  Segments:
    music        0.0s -> 4.4s (4.4s)
    noEnergy     4.4s -> 4.8s (0.4s)
    female       4.8s -> 7.1s (2.3s)
    noise        7.1s -> 19.7s (12.6s)
    female       19.7s -> 23.4s (3.7s)
    noise        23.4s -> 30.0s (6.6s)
  Summary:
    Male   : 0.0s (0.0%)
    Female : 6.0s (100.0%)
    Result : FEMALE
42/42 - 1s - 17ms/step
33/33 - 1s - 38ms/step


Gender detection:  11%|█         | 2/18 [00:17<02:02,  7.66s/it]

...


Processing: DevikaGupta
  Segments:
    music        0.0s -> 4.6s (4.6s)
    female       4.6s -> 9.5s (4.9s)
    noEnergy     9.5s -> 10.2s (0.7s)
    music        10.2s -> 17.6s (7.5s)
    female       17.6s -> 30.0s (12.4s)
  Summary:
    Male   : 0.0s (0.0%)
    Female : 17.3s (100.0%)
    Result : FEMALE
46/46 - 1s - 15ms/step
8/8 - 0s - 7ms/step


Gender detection:  72%|███████▏  | 13/18 [00:42<00:10,  2.01s/it]


Processing: DevikaGupta
  Segments:
    music        0.0s -> 8.6s (8.6s)
    female       8.6s -> 10.4s (1.8s)
    male         10.4s -> 11.6s (1.3s)
    female       11.6s -> 13.5s (1.8s)
    music        13.5s -> 18.1s (4.7s)
    noEnergy     18.1s -> 19.1s (0.9s)
    music        19.1s -> 30.0s (10.9s)
  Summary:
    Male   : 1.3s (25.9%)
    Female : 3.6s (74.1%)
    Result : FEMALE


Gender detection:  72%|███████▏  | 13/18 [00:43<00:16,  3.36s/it]


KeyboardInterrupt: 

### CELL 9 (optional): Clean up local disk

In [ ]:
# Run this only after you've confirmed RESULTS_CSV on Drive looks correct.
# It just clears Colab's temporary disk, not your actual audio files.
import shutil
shutil.rmtree(TRIMMED_DIR, ignore_errors=True)
# print("Cleaned up temporary trimmed audio.")

### CELL 10 (diagnostic): Listen to one sample before retrying everything

Run this, download `/content/sample_check.mp3` from the Colab file browser
(left sidebar), and actually listen. Confirms this is really an intro/silence
issue before spending compute time on the full retry below.

In [ ]:
current_df = pd.read_csv(RESULTS_CSV)
unknown_files = current_df[current_df["dominant_gender"] == "unknown"]["file"].tolist()
print(f"{len(unknown_files)} files currently marked unknown")

unknown_check = current_df[current_df["dominant_gender"] == "unknown"].copy()
unknown_check["channel"] = unknown_check["file"].apply(lambda f: f.split("__")[0] if "__" in f else f)
print("\nTop affected channels:")
print(unknown_check["channel"].value_counts().head(10))

CHECK_CHANNEL = "ShibuThomas"  # change to whichever channel you want to spot-check
sample_files = [f for f in unknown_files if CHECK_CHANNEL in f]
if sample_files:
    sample_path = glob.glob(os.path.join(AUDIO_DIR, "**", sample_files[0]), recursive=True)
    if sample_path:
        import shutil
        shutil.copy(sample_path[0], "/content/sample_check.mp3")
        print(f"\nCopied {sample_files[0]} to /content/sample_check.mp3 - download and listen")
    else:
        print("File not found on disk - check AUDIO_DIR path")
else:
    print(f"No unknown files found for channel {CHECK_CHANNEL}")

970 files currently marked unknown

Top affected channels:
channel
Chitralekhaji            74
MilikyaMili              72
BhajanMarg               70
KrutikaPlays             64
MohanCLazarus            58
GurudevHindi             55
GoldyHindiGaming         45
KnowledgeGate            41
AnkkitaC                 40
AnkitSajwanMinistries    34
Name: count, dtype: int64

Copied ShibuThomasOfficial__9nndHdDvzB8.mp3 to /content/sample_check.mp3 - download and listen


### CELL 11: Retry unknown files using seconds 30-90, update RESULTS_CSV in place

Extracts a fresh 60-second window starting at 30s (skipping the already-tried
0-30s window), reruns detection, and **updates** existing rows rather than
adding duplicates.

In [ ]:
RETRY_START = 30
RETRY_DURATION = 60

retry_results = []
still_unknown = []

for fname in tqdm(unknown_files, desc="Retrying with later window"):
    matches = glob.glob(os.path.join(AUDIO_DIR, "**", fname), recursive=True)
    if not matches:
        continue
    src_path = matches[0]
    retry_clip = f"/content/retry_{fname}"

    try:
        cmd = ["ffmpeg", "-y", "-i", src_path, "-ss", str(RETRY_START), "-t", str(RETRY_DURATION), retry_clip]
        subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, timeout=60)

        segments = seg(retry_clip)
        male_time = sum(end - start for label, start, end in segments if label == "male")
        female_time = sum(end - start for label, start, end in segments if label == "female")
        total_speech = male_time + female_time

        if total_speech > 0:
            dominant = "male" if male_time >= female_time else "female"
            retry_results.append({
                "file": fname, "male_seconds": round(male_time, 2),
                "female_seconds": round(female_time, 2), "dominant_gender": dominant
            })
        else:
            still_unknown.append(fname)

        os.remove(retry_clip)
    except Exception as e:
        print(f"Error on {fname}: {e}")
        still_unknown.append(fname)

print(f"\nRecovered: {len(retry_results)} files now have a gender label")
print(f"Still unknown even after retry: {len(still_unknown)} files")

# update RESULTS_CSV: drop old unknown rows for recovered files, add new results
retry_df = pd.DataFrame(retry_results)
recovered_files = set(retry_df["file"]) if len(retry_df) else set()

df_updated = pd.read_csv(RESULTS_CSV)
df_updated = df_updated[~df_updated["file"].isin(recovered_files)]
df_updated = pd.concat([df_updated, retry_df], ignore_index=True)
df_updated.to_csv(RESULTS_CSV, index=False)

print(f"\nUpdated {RESULTS_CSV}")
print("New gender distribution:")
print(df_updated["dominant_gender"].value_counts())

if still_unknown:
    still_unknown_path = os.path.join(os.path.dirname(RESULTS_CSV), "still_unknown_after_retry.csv")
    pd.Series(still_unknown, name="file").to_csv(still_unknown_path, index=False)
    print(f"\n{len(still_unknown)} files remain unknown even after retry - saved to {still_unknown_path}")

Retrying with later window:   0%|          | 0/970 [00:00<?, ?it/s]

90/90 - 1s - 11ms/step


Retrying with later window:   0%|          | 1/970 [00:04<1:05:50,  4.08s/it]

67/67 - 0s - 4ms/step
10/10 - 1s - 130ms/step


Retrying with later window:   0%|          | 2/970 [00:09<1:22:30,  5.11s/it]

...

85/85 - 0s - 3ms/step
66/66 - 0s - 6ms/step


Retrying with later window: 100%|█████████▉| 969/970 [1:00:58<00:04,  4.33s/it]

94/94 - 0s - 3ms/step
13/13 - 0s - 8ms/step


Retrying with later window: 100%|██████████| 970/970 [1:01:01<00:00,  3.77s/it]


Recovered: 494 files now have a gender label
Still unknown even after retry: 473 files

Updated /content/drive/MyDrive/colab_notebooks/masculine_default/gender_results.csv
New gender distribution:
dominant_gender
male       5564
female     3868
unknown     476
Name: count, dtype: int64

473 files remain unknown even after retry - saved to /content/drive/MyDrive/colab_notebooks/masculine_default/still_unknown_after_retry.csv


### CELL 12: Summary — how many are still unknown after the retry

Checks `still_unknown_after_retry.csv` (saved by Cell 11) against the live
`RESULTS_CSV`, so you can see exactly how many videos remain unresolved and
which channels they belong to — useful for deciding if a second retry with
an even later window is worth it, or if it's small enough to just note as
a limitation in the paper.

In [ ]:
still_unknown_path = os.path.join(os.path.dirname(RESULTS_CSV), "still_unknown_after_retry.csv")

if os.path.exists(still_unknown_path):
    still_unknown_df = pd.read_csv(still_unknown_path)
    print(f"Total still unknown after retry: {len(still_unknown_df)}")

    still_unknown_df["channel"] = still_unknown_df["file"].apply(lambda f: f.split("__")[0] if "__" in f else f)
    print("\nBy channel:")
    print(still_unknown_df["channel"].value_counts().to_string())
else:
    print("No file found - either the retry hasn't been run yet, or nothing remained unknown after it")

# also show the final overall count directly from the results CSV, as a cross-check
final_df = pd.read_csv(RESULTS_CSV)
print(f"\nCross-check - current gender_results.csv breakdown:")
print(final_df["dominant_gender"].value_counts())
print(f"\nUnknown as % of total: {(final_df['dominant_gender']=='unknown').sum() / len(final_df) * 100:.1f}%")

Total still unknown after retry: 473

By channel:
channel
Chitralekhaji                                   62
KrutikaPlays                                    59
GurudevHindi                                    51
BhajanMarg                                      44
MilikyaMili                                     43
AnkkitaC                                        39
Terapanth                                       25
AnkitSajwanMinistries                           25
YasminBodyImage                                 20
KnowledgeGate                                   12
Aniruddhacharyaji                               11
MohanCLazarus                                    8
JaiJinendra                                      8
ShibuThomasOfficial                              7
GoldyHindiGaming                                 6
CarryisLive                                      6
FilterCopy                                       5
GodLikeEsports                                   5
PayalGaming             